# 00. 환경 설정 및 설치 확인

**이 Notebook은 처음 한 번만 실행하면 됩니다.**

순서:
1. 필요한 라이브러리 설치
2. 설치 확인
3. `.env` 파일 생성 (API 키 저장)
4. OpenAI API 연결 테스트
5. SQLite DB 초기화

## 1단계: 라이브러리 설치

아래 셀을 실행하면 필요한 패키지가 모두 설치됩니다. (2~5분 소요)

In [1]:
import subprocess, sys

packages = [
    'voila',
    'pandas',
    'numpy',
    'plotly',
    'ipywidgets',
    'panel',
    'finance-datareader',
    'pykrx',
    'openai',
    'python-dotenv',
    'pyyaml',
    'nbconvert',
    'apscheduler',
]

for pkg in packages:
    print(f'설치 중: {pkg} ...', end=' ')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print('완료')
    else:
        print(f'오류 — {result.stderr.strip()}')

print('\n모든 설치 완료!')

설치 중: voila ... 완료
설치 중: pandas ... 완료
설치 중: numpy ... 완료
설치 중: plotly ... 완료
설치 중: ipywidgets ... 완료
설치 중: panel ... 완료
설치 중: finance-datareader ... 완료
설치 중: pykrx ... 완료
설치 중: openai ... 완료
설치 중: python-dotenv ... 완료
설치 중: pyyaml ... 완료
설치 중: nbconvert ... 완료
설치 중: apscheduler ... 완료

모든 설치 완료!


## 2단계: 설치 확인

In [2]:
import sys
print(f'Python 버전: {sys.version}')
print()

checks = [
    ('pandas',            'pandas'),
    ('numpy',             'numpy'),
    ('plotly',            'plotly'),
    ('ipywidgets',        'ipywidgets'),
    ('FinanceDataReader', 'FinanceDataReader'),
    ('pykrx',             'pykrx'),
    ('openai',            'openai'),
    ('dotenv',            'dotenv'),
    ('yaml',              'yaml'),
    ('apscheduler',       'apscheduler'),
]

all_ok = True
for name, mod in checks:
    try:
        __import__(mod)
        print(f'  OK  {name}')
    except ImportError:
        print(f'  FAIL  {name}  ← 1단계를 다시 실행하세요')
        all_ok = False

print()
if all_ok:
    print('모든 라이브러리 정상 설치되었습니다!')
else:
    print('일부 라이브러리 설치에 문제가 있습니다. 위에서 FAIL 항목을 확인하세요.')

Python 버전: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]

  OK  pandas
  OK  numpy
  OK  plotly
  OK  ipywidgets
  OK  FinanceDataReader
KRX 로그인 실패: KRX_ID 또는 KRX_PW 환경 변수가 설정되지 않았습니다.
  OK  pykrx
  OK  openai
  OK  dotenv
  OK  yaml
  OK  apscheduler

모든 라이브러리 정상 설치되었습니다!


## 3단계: .env 파일 생성

`.env.template` 파일을 복사해서 `.env`를 만들고 OpenAI API 키를 입력합니다.

아래 셀을 실행하면 `.env` 파일이 없을 때 자동으로 템플릿에서 복사합니다.

In [3]:
import os, shutil
from pathlib import Path

# 현재 Notebook 위치를 프로젝트 루트로 설정
PROJECT_ROOT = Path.cwd()
env_file     = PROJECT_ROOT / '.env'
env_template = PROJECT_ROOT / '.env.template'

if env_file.exists():
    print('.env 파일이 이미 있습니다.')
elif env_template.exists():
    shutil.copy(env_template, env_file)
    print('.env 파일을 생성했습니다.')
    print()
    print('지금 .env 파일을 열어서 아래 항목을 채워주세요:')
    print('  OPENAI_API_KEY=sk-여기에_실제_키_입력')
    print('  GMAIL_ADDRESS=jonghun.kim84@gmail.com')
    print('  GMAIL_APP_PASSWORD=Gmail_앱_비밀번호')
else:
    print('.env.template 파일을 찾을 수 없습니다. 프로젝트 폴더에서 실행하고 있는지 확인하세요.')

.env 파일이 이미 있습니다.


## 4단계: OpenAI API 연결 테스트

`.env` 파일에 API 키를 입력한 뒤 아래 셀을 실행하세요.

In [4]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY', '')

if not api_key or api_key.startswith('sk-여기에'):
    print('.env 파일에 실제 OpenAI API 키를 입력해 주세요.')
else:
    try:
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': '"안녕하세요" 라고만 답해주세요.'}],
            max_tokens=20
        )
        print('OpenAI API 연결 성공!')
        print(f'응답: {response.choices[0].message.content}')
    except Exception as e:
        print(f'연결 실패: {e}')
        print('API 키가 올바른지 확인하세요.')

OpenAI API 연결 성공!
응답: 안녕하세요.


## 5단계: SQLite DB 초기화

이력을 저장할 로컬 데이터베이스를 만듭니다.

In [5]:
import sqlite3
from pathlib import Path

db_path = Path.cwd() / 'portfolio.db'
conn = sqlite3.connect(db_path)
cur = conn.cursor()

# 버킷 스냅샷 테이블
cur.execute('''
CREATE TABLE IF NOT EXISTS bucket_snapshots (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    date        TEXT NOT NULL,
    bucket1     REAL DEFAULT 0,
    bucket2     REAL DEFAULT 0,
    bucket3     REAL DEFAULT 0,
    total       REAL DEFAULT 0
)
''')

# 위험 점수 테이블
cur.execute('''
CREATE TABLE IF NOT EXISTS risk_scores (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    date        TEXT NOT NULL,
    total_score REAL DEFAULT 0,
    cash_score  REAL DEFAULT 0,
    seq_score   REAL DEFAULT 0,
    conc_score  REAL DEFAULT 0,
    level       TEXT DEFAULT 'green'
)
''')

# AI 제안 이력 테이블
cur.execute('''
CREATE TABLE IF NOT EXISTS recommendations (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    date        TEXT NOT NULL,
    rule_id     TEXT,
    message     TEXT,
    status      TEXT DEFAULT 'pending'
)
''')

# 인출 이력 테이블
cur.execute('''
CREATE TABLE IF NOT EXISTS withdrawal_log (
    id                INTEGER PRIMARY KEY AUTOINCREMENT,
    date              TEXT NOT NULL,
    amount            REAL DEFAULT 0,
    guardrail_applied INTEGER DEFAULT 0,
    note              TEXT
)
''')

# 알림 발송 이력
cur.execute('''
CREATE TABLE IF NOT EXISTS alert_log (
    id       INTEGER PRIMARY KEY AUTOINCREMENT,
    date     TEXT NOT NULL,
    channel  TEXT,
    subject  TEXT,
    sent_ok  INTEGER DEFAULT 0
)
''')

conn.commit()
conn.close()

print(f'DB 초기화 완료: {db_path}')
print('생성된 테이블: bucket_snapshots, risk_scores, recommendations, withdrawal_log, alert_log')

DB 초기화 완료: C:\Users\kwak5\OneDrive\바탕 화면\은퇴포트폴리오AI\portfolio.db
생성된 테이블: bucket_snapshots, risk_scores, recommendations, withdrawal_log, alert_log


## 6단계: config.yaml 확인

In [6]:
import yaml
from pathlib import Path

config_path = Path.cwd() / 'config.yaml'

if not config_path.exists():
    print('config.yaml 파일이 없습니다. 프로젝트 폴더를 확인하세요.')
else:
    with open(config_path, encoding='utf-8') as f:
        config = yaml.safe_load(f)

    print('config.yaml 로드 성공!')
    print()
    print(f"  이름: {config['user']['name']}")
    print(f"  출생년도: {config['user']['birth_year']}")
    print(f"  월 생활비: {config['user']['monthly_expense']:,}원")
    print(f"  위험 성향: {config['user']['risk_profile']}")
    print()
    print('목표 포트폴리오 비중:')
    p = config['portfolio']
    print(f"  현금성:  {p['target_cash']*100:.0f}%")
    print(f"  채권:    {p['target_bond']*100:.0f}%")
    print(f"  주식형:  {p['target_equity']*100:.0f}%")
    print(f"  인컴:    {p['target_income']*100:.0f}%")
    total = p['target_cash']+p['target_bond']+p['target_equity']+p['target_income']
    print(f"  합계:    {total*100:.0f}% {'OK' if abs(total-1.0)<0.001 else '← 합계가 100%가 아닙니다. 확인하세요.'}")

config.yaml 로드 성공!

  이름: 종헌
  출생년도: 1965
  월 생활비: 2,500,000원
  위험 성향: balanced

목표 포트폴리오 비중:
  현금성:  25%
  채권:    25%
  주식형:  35%
  인컴:    15%
  합계:    100% OK


---

## 설정 완료!

모든 셀이 정상적으로 실행되었다면 다음 단계로 이동하세요.

**다음: `01_data_input.ipynb` — 보유 자산 입력**